# Modular Fair-RAG Recreation (T5 Small + BM25)

This notebook is designed to scale across machines by changing only one config profile.

Profiles:
- `weak`: 5 queries, small samples
- `balanced`: more queries and samples
- `strong`: intended for a better machine/full recreation

Output focus: normalized EE-D interval buckets (`0.0-0.2 ... 0.8-1.0`) and EU differences.

In [28]:
from pathlib import Path
import importlib
import pandas as pd

import notebook_experiment_utils as neu
importlib.reload(neu)

ExperimentConfig = neu.ExperimentConfig
ensure_paths_exist = neu.ensure_paths_exist
run_gold_baseline = neu.run_gold_baseline
run_retriever_grid = neu.run_retriever_grid
run_deterministic_reference = neu.run_deterministic_reference
run_mmr_deterministic = neu.run_mmr_deterministic
normalize_eu_grid = neu.normalize_eu_grid
load_normalized_rows = neu.load_normalized_rows
load_raw_rows = neu.load_raw_rows
add_ee_d_interval_bins = neu.add_ee_d_interval_bins
summarize_by_interval = neu.summarize_by_interval
interval_pvalues_vs_deterministic = neu.interval_pvalues_vs_deterministic
save_interval_outputs = neu.save_interval_outputs
save_per_query_log = neu.save_per_query_log
save_pvalue_outputs = neu.save_pvalue_outputs
reset_run_outputs = neu.reset_run_outputs
assert_consistent_qids_across_alphas = neu.assert_consistent_qids_across_alphas


In [29]:
ROOT = Path.cwd()
PY = ROOT / '.venv' / 'Scripts' / 'python.exe'

profiles = {
    'weak': {
        'max_queries': 30,
        'n_samples': 10,
        'k': 5,
    },
    'balanced': {
        'max_queries': 50,
        'n_samples': 20,
        'k': 5,
    },
    'strong': {
        'max_queries': None,
        'n_samples': 100,
        'k': 5,
    },
}

# Change only this line when moving to a stronger machine
profile_name = 'weak'
profile = profiles[profile_name]

cfg = ExperimentConfig(
    root=ROOT,
    python_exe=PY,
    generator_name='flanT5Small',
    lamp_num=4,
    retriever_name='bm25',
    alphas=(1, 2, 4, 8),
    max_queries=profile['max_queries'],
    n_samples=profile['n_samples'],
    k=profile['k'],
    remove_temp_files=True,
    skip_existing=True,
    mmr_base_retriever='bm25',
    mmr_lambda=0.7,
)

ensure_paths_exist([cfg.python_exe])

print('Profile:', profile_name)
print('Config :', cfg)

Profile: weak
Config : ExperimentConfig(root=WindowsPath('c:/code/Fair-RAG/Fair-RAG'), python_exe=WindowsPath('c:/code/Fair-RAG/Fair-RAG/.venv/Scripts/python.exe'), generator_name='flanT5Small', lamp_num=4, retriever_name='bm25', alphas=(1, 2, 4, 8), max_queries=30, n_samples=10, k=5, remove_temp_files=True, skip_existing=True, mmr_base_retriever='bm25', mmr_lambda=0.7)


## Run Controls
Toggle stages independently for iterative work or resume mode.

In [24]:
RUN_GOLD = True
RUN_BM25_GRID = True
RUN_DETERMINISTIC_REF = True
RUN_MMR_DETERMINISTIC = True
RUN_NORMALIZE_EU = True
RUN_ANALYSIS = True

# Deterministic references use one alpha only; value is for naming consistency.
DETERMINISTIC_ALPHA = 1

# Set to True when switching profiles or after interrupted/partial runs.
FORCE_FRESH_RUN = True

In [25]:
if FORCE_FRESH_RUN:
    reset_run_outputs(cfg, include_gold=True)

if RUN_GOLD:
    run_gold_baseline(cfg, alpha=8)
else:
    print('Skipping gold baseline')

if RUN_BM25_GRID:
    run_retriever_grid(cfg)
else:
    print('Skipping BM25 alpha grid')

if RUN_DETERMINISTIC_REF:
    run_deterministic_reference(cfg, alpha=DETERMINISTIC_ALPHA, output_suffix='_deterministic')
else:
    print('Skipping deterministic reference')

if RUN_MMR_DETERMINISTIC:
    run_mmr_deterministic(cfg, alpha=DETERMINISTIC_ALPHA, output_suffix='_mmr_deterministic')
else:
    print('Skipping MMR deterministic run')

if RUN_NORMALIZE_EU:
    # Prevent KeyError in normalize_eu.py due to mismatched qids across alpha files.
    assert_consistent_qids_across_alphas(cfg, cfg.retriever_name)
    normalize_eu_grid(cfg)
else:
    print('Skipping EU normalization')

reset_run_outputs: deleted 9 file(s)

> c:\code\Fair-RAG\Fair-RAG\.venv\Scripts\python.exe experiment.py --generator_name flanT5Small --lamp_num 4 --retriever_name gold --alpha 8 --k 5 --n_samples 5 --max_queries 15 --remove_temp_files
[10/15] avg EE-D: 0.5824 | avg EE-R: 0.9921 | avg EU (rouge-l): 0.0535
[15/15] FINAL avg EE-D: 0.5488 | avg EE-R: 0.9922 | avg EU (rouge-l): 0.0630


> c:\code\Fair-RAG\Fair-RAG\.venv\Scripts\python.exe experiment.py --generator_name flanT5Small --lamp_num 4 --retriever_name bm25 --alpha 1 --k 5 --n_samples 5 --max_queries 15 --remove_temp_files
[10/15] avg EE-D: 0.2880 | avg EE-R: 0.1870 | avg EU (rouge-l): 0.0175
[15/15] FINAL avg EE-D: 0.2725 | avg EE-R: 0.1806 | avg EU (rouge-l): 0.0194


> c:\code\Fair-RAG\Fair-RAG\.venv\Scripts\python.exe experiment.py --generator_name flanT5Small --lamp_num 4 --retriever_name bm25 --alpha 2 --k 5 --n_samples 5 --max_queries 15 --remove_temp_files
[10/15] avg EE-D: 0.3392 | avg EE-R: 0.1846 | avg EU (rouge-l): 0.02

In [30]:
run_mmr_deterministic(cfg, alpha=DETERMINISTIC_ALPHA, output_suffix='_mmr_deterministic')


> c:\code\Fair-RAG\Fair-RAG\.venv\Scripts\python.exe experiment.py --generator_name flanT5Small --lamp_num 4 --retriever_name mmr --alpha 1 --k 5 --n_samples 1 --deterministic_ranking --mmr_base_retriever bm25 --mmr_lambda 0.7 --output_suffix _mmr_deterministic --max_queries 30 --remove_temp_files
[10/30] avg EE-D: 1.0000 | avg EE-R: 0.1400 | avg EU (rouge-l): 0.0285
[20/30] avg EE-D: 1.0000 | avg EE-R: 0.1340 | avg EU (rouge-l): 0.0278
[30/30] avg EE-D: 1.0000 | avg EE-R: 0.1827 | avg EU (rouge-l): 0.0238



## Publish Results by EE-D Interval

In [ ]:
if RUN_ANALYSIS:
    # Main published table: normalized BM25 alpha-grid metrics.
    df = load_normalized_rows(cfg)
    df = add_ee_d_interval_bins(df)
    summary = summarize_by_interval(df)

    print(
        f'Analyzing retriever: {cfg.retriever_name} '
        '(oracle gold excluded; used only as normalization reference)'
    )
    print('Per-query rows (BM25 alpha grid):', len(df))
    display(summary)

    # Deterministic BM25 reference results (raw scale).
    df_det = load_raw_rows(
        cfg,
        cfg.retriever_name,
        (DETERMINISTIC_ALPHA,),
        output_suffix='_deterministic',
    )
    df_det = add_ee_d_interval_bins(df_det)
    summary_det = summarize_by_interval(df_det)
    print('\nDeterministic BM25 reference (raw EE/EU):')
    display(summary_det)

    # Deterministic MMR run (raw scale).
    df_mmr = load_raw_rows(
        cfg,
        'mmr',
        (DETERMINISTIC_ALPHA,),
        output_suffix='_mmr_deterministic',
    )
    df_mmr = add_ee_d_interval_bins(df_mmr)
    summary_mmr = summarize_by_interval(df_mmr)
    print('\nDeterministic MMR run (raw EE/EU):')
    display(summary_mmr)

    out_name = f'notebook_outputs_{profile_name}'
    summary_fp = save_interval_outputs(cfg, summary, out_name)
    perq_fp = save_per_query_log(cfg, df, out_name)

    # Simple significance view: MMR deterministic vs BM25 deterministic, p-value per interval.
    pvals_ee_r = interval_pvalues_vs_deterministic(df_mmr, df_det, metric='ee_r')
    pvals_eu = interval_pvalues_vs_deterministic(df_mmr, df_det, metric='eu')
    pvals = pd.concat([pvals_ee_r, pvals_eu], ignore_index=True)

    # Keep only intervals present in MMR run.
    pvals = pvals[pvals['n_qids'] > 0].copy()
    pvals = (
        pvals[[
            'ee_d_interval',
            'metric',
            'n_qids',
            'p_two_sided',
        ]]
        .sort_values(['ee_d_interval', 'metric'])
        .reset_index(drop=True)
    )
    display(pvals)
    pvals_fp = save_pvalue_outputs(cfg, pvals, out_name)

    print('Saved:')
    print(' -', summary_fp)
    print(' -', perq_fp)
    print(' -', pvals_fp)
else:
    print('Skipping analysis')


Analyzing retriever: bm25 (oracle gold excluded; used only as normalization reference)
Per-query rows: 60


,ee_d_interval,n_queries,mean_eu,mean_ee_d,mean_ee_r
0,0.2-0.4,13,0.169016,0.266462,0.127224
1,0.4-0.6,3,0.320062,0.472000,0.422472
2,0.6-0.8,8,0.290915,0.718000,0.230106
3,0.8-1.0,15,0.204278,0.935273,0.158221


,alpha,ee_d_interval,metric,n_qids,mean_diff_st_minus_det,p_two_sided,p_st_greater,p_st_less
0,1,0.2-0.4,ee_r,13,-0.022399,0.792969,0.628906,0.396484
1,1,0.4-0.6,ee_r,2,0.260000,0.500000,0.250000,1.000000
2,1,0.6-0.8,ee_r,0,NaN,NaN,NaN,NaN
3,1,0.8-1.0,ee_r,0,NaN,NaN,NaN,NaN
4,2,0.2-0.4,ee_r,13,-0.012514,1.000000,0.507324,0.507324
5,2,0.4-0.6,ee_r,1,NaN,NaN,NaN,NaN
6,2,0.6-0.8,ee_r,1,NaN,NaN,NaN,NaN
7,2,0.8-1.0,ee_r,0,NaN,NaN,NaN,NaN
8,4,0.2-0.4,ee_r,0,NaN,NaN,NaN,NaN
9,4,0.4-0.6,ee_r,1,NaN,NaN,NaN,NaN


Saved:
 - c:\code\Fair-RAG\Fair-RAG\experiment_results\flanT5Small\lamp4\bm25\notebook_outputs_weak\ee_d_interval_summary.csv
 - c:\code\Fair-RAG\Fair-RAG\experiment_results\flanT5Small\lamp4\bm25\notebook_outputs_weak\per_query_all_metrics.csv
 - c:\code\Fair-RAG\Fair-RAG\experiment_results\flanT5Small\lamp4\bm25\notebook_outputs_weak\pvalues_vs_deterministic_by_interval.csv
